# Session 8 — Your turn: Retrieval, tools and agents

Write the two retrieval ingredients that need no model: the splitting of a document into passages, and the fusion of two rankings into one.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import re
import numpy as np
from aij_check import check, close, summary

## Task 1: Chunk a text into passages

Write `chunk(text, max_words)`: split the text into sentences (after a full stop, question mark or exclamation mark followed by whitespace), and gather whole sentences into passages of at most `max_words` words each. A sentence is never split; a sentence longer than `max_words` becomes a passage of its own.
Return a list of strings.

In [2]:
def chunk(text, max_words):
    passages, current = [], []
    for sentence in re.split(r"(?<=[.!?])\s+", text.strip()):
        if current and len(" ".join(current + [sentence]).split()) > max_words:
            passages.append(" ".join(current)); current = []
        current.append(sentence)
    if current: passages.append(" ".join(current))
    return passages


text = ("Retrieval finds the passages that match a question. A small model embeds each passage. The question is embedded too! "
        "Then the closest passages are put in the prompt. Does the model use them? Usually, if they contain the answer.")

In [3]:
check("a large limit gives one passage", lambda: len(chunk(text, 100)) == 1)
check("a small limit gives more, shorter passages", lambda: len(chunk(text, 12)) > len(chunk(text, 25)) >= 2)
check("no passage exceeds the limit unless it is a single sentence", lambda: all(len(p.split()) <= 12 or len(re.split(r"(?<=[.!?])\s+", p)) == 1 for p in chunk(text, 12)))
check("nothing is lost: the passages joined are the text", lambda: " ".join(chunk(text, 12)) == text)
summary()

PASS  a large limit gives one passage
PASS  a small limit gives more, shorter passages
PASS  no passage exceeds the limit unless it is a single sentence
PASS  nothing is lost: the passages joined are the text

4 of 4 checks passed.  Well done.


**Why it works.** Smaller passages are more precise but may cut an answer away from the words of the question; larger ones keep context but dilute the match and fill the prompt. The session measures Recall@3 at 60, 120 and 250 words and finds no single best size.

## Task 2: Reciprocal rank fusion

Two retrievers ranked five passages (best first): `keyword = ["p1", "p2", "p3", "p4", "p5"]` and `semantic = ["p3", "p5", "p1", "p2", "p4"]`.
Write `rank_fusion(rankings, k=60)` that gives each passage the score `sum(1 / (k + rank))` over the rankings (rank counted from 1) and returns the passages ordered by score, best first. Store the result for the two rankings in `fused`.

In [4]:
keyword = ["p1", "p2", "p3", "p4", "p5"]
semantic = ["p3", "p5", "p1", "p2", "p4"]


def rank_fusion(rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, passage in enumerate(ranking, start=1):
            score[passage] = score.get(passage, 0.0) + 1 / (k + rank)
    return sorted(score, key=lambda p: -score[p])


fused = rank_fusion([keyword, semantic])
print(fused)

['p1', 'p3', 'p2', 'p5', 'p4']


In [5]:
check("all five passages appear once", lambda: sorted(fused) == ["p1", "p2", "p3", "p4", "p5"])
check("p3 (3rd and 1st) beats p1 (1st and 3rd) by a hair or ties, and both beat p4", lambda: fused.index("p3") < fused.index("p4") and fused.index("p1") < fused.index("p4"))
check("p5 (5th and 2nd) is ranked after p3 and p1", lambda: fused.index("p5") > fused.index("p3") and fused.index("p5") > fused.index("p1"))
check("a passage first in one list and absent from another still scores", lambda: rank_fusion([["a", "b"], ["b"]])[0] == "b")
summary()

PASS  all five passages appear once
PASS  p3 (3rd and 1st) beats p1 (1st and 3rd) by a hair or ties, and both beat p4
PASS  p5 (5th and 2nd) is ranked after p3 and p1
PASS  a passage first in one list and absent from another still scores

4 of 4 checks passed.  Well done.


**Why it works.** Fusion needs only ranks, not scores, so it can combine retrievers whose scores are on different scales (TF-IDF and cosine similarity). A passage ranked well by both beats one ranked first by only one, which is why the fused retriever is the safest choice in the session.